# Multi-Robot Warehouse Coordination Demo

This notebook demonstrates the `multi_robot_coordination` task added for the
Meta PyTorch OpenEnv Hackathon 2026 Grand Finale.

Two robots operate **concurrently** on a shared order queue:
- Each robot has its own position and inventory.
- They share the same grid, obstacles, and order pool.
- A collision-avoidance penalty keeps them apart.
- A coordination-efficiency bonus rewards balanced workload.

**Action space (per robot):** `0` up | `1` down | `2` left | `3` right | `4` pick | `5` deliver

In [ ]:
import sys, os
# Make sure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from warehouse_env.server.multi_robot_environment import MultiRobotWarehouse
from warehouse_env.client import reset_multi_robot, step_multi_robot

print('✓ Multi-robot environment imported successfully')
print(f'  Robots available: R1, R2')

## 1. Initialize and Reset

In [ ]:
# Reset on simple_order task (5x5 grid, 1 order, 2 items)
env, obs = reset_multi_robot(task_name='simple_order', num_robots=2)

print(f'Grid size : {env.game_state.grid_size}')
print(f'Max steps : {env.game_state.max_steps}')
print(f'Obstacles : {env.game_state.obstacles}')
print(f'Orders    : {env.game_state.total_orders_expected}')
print()
for rid, o in obs.items():
    print(f'  {rid} starting at {o.robot_pos}')
print()
# Show the initial grid
print(obs['R1'].render)

## 2. Run a Random Policy Episode

In [ ]:
import random

env, obs = reset_multi_robot(task_name='simple_order', num_robots=2)

total_reward = 0.0
step = 0
done = False

while not done:
    # Random action for each robot
    actions = {rid: random.randint(0, 5) for rid in env.robots}
    env, obs, reward, done = step_multi_robot(env, actions)
    total_reward += reward
    step += 1

print(f'Episode finished in {step} steps')
print(f'Total reward    : {total_reward:.4f}')
print(f'Final score     : {env.game_state.score:.4f}')
print(f'Orders completed: {len(env.game_state.completed_orders)}/{env.game_state.total_orders_expected}')
print(f'Collisions      : {env._collision_count}')
print()
print('Final grid:')
print(obs['R1'].render)

## 3. Evaluate Across All Task Tiers

In [ ]:
TASKS = ['simple_order', 'multi_step_order', 'order_queue', 'adaptive_fulfillment']
N_EPISODES = 3

results = {}

for task in TASKS:
    scores, collisions_list, completions = [], [], []
    for ep in range(N_EPISODES):
        env, obs = reset_multi_robot(task_name=task, num_robots=2)
        done = False
        while not done:
            actions = {rid: random.randint(0, 5) for rid in env.robots}
            env, obs, _, done = step_multi_robot(env, actions)
        scores.append(env.game_state.score)
        collisions_list.append(env._collision_count)
        completions.append(
            len(env.game_state.completed_orders) / max(env.game_state.total_orders_expected, 1)
        )
    results[task] = {
        'avg_score': sum(scores) / len(scores),
        'avg_collisions': sum(collisions_list) / len(collisions_list),
        'avg_completion': sum(completions) / len(completions),
    }

print(f'{"Task":<25} {"Avg Score":>10} {"Avg Collisions":>15} {"Completion":>12}')
print('-' * 65)
for task, r in results.items():
    print(f"{task:<25} {r['avg_score']:>10.4f} {r['avg_collisions']:>15.1f} {r['avg_completion']:>12.2%}")

## 4. Coordination Insight: R1 vs R2 Step Counts

In [ ]:
env, obs = reset_multi_robot(task_name='order_queue', num_robots=2)
done = False
while not done:
    actions = {rid: random.randint(0, 5) for rid in env.robots}
    env, obs, _, done = step_multi_robot(env, actions)

print('Workload distribution (order_queue task):')
for rid, robot in env.robots.items():
    print(f'  {rid}: {robot.steps_taken} steps, {robot.collisions} collisions')
print(f'  Total collisions : {env._collision_count}')
print(f'  Final score      : {env.game_state.score:.4f}')

## Summary

The `MultiRobotWarehouse` environment demonstrates:

| Feature | Details |
|---|---|
| **Concurrency** | Both robots act simultaneously each step |
| **Collision avoidance** | If two robots target the same cell, neither moves and a penalty is applied |
| **Shared order queue** | Orders are pre-assigned to idle robots; reassignment happens automatically |
| **Coordination score** | Penalises workload imbalance between robots |
| **Full OpenEnv compat** | Uses same `WarehouseObservation` / `WarehouseGameState` models as the single-robot env |

This is registered as task `multi_robot_coordination` in `openenv.yaml`.